In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import zipfile  # noqa: E402
import numpy as np  # noqa: E402
import numpy.lib.format as npformat  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from scipy.stats import zscore  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.analysis.pca_polarity import (  # noqa: E402
    align_pc1_signs,
    apply_pc1_signs,
    topography_consistency,
)
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    MusicTypeVariants,
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Wavelet Power — Channel-PCA Time-Frequency Maps

Reduce the stimulus-locked **wavelet power** to a small number of **time-frequency
(TF) maps per participant** by collapsing the channel dimension with **PCA over
channels**, following the same data-access workflow as
`assr_stimulus_aligned_inspection.ipynb`.

The first `N_COMPONENTS` components (default **3**) are extracted and treated
**identically** — same trial average, same cross-participant polarity alignment,
same diagnostics, same plots — so PC2/PC3 can be compared against PC1 instead of
being assumed uninformative. This mirrors `assr_raw_pca_analysis.ipynb`. The
dominant-variance component is not necessarily the best 40 Hz carrier: PCA ranks
components by explained *channel variance* over the whole TF plane, which a broad
low-frequency onset response can dominate while the narrow 40 Hz steady-state ends
up in a later component.

Pipeline (per subject, per project convention on the **Placebo** condition):

1. **Trial average.** Epoch the wavelet power around every `fam+` onset and
   average over stimuli. The epoch is **0.1 s before onset to 1.0 s after** it —
   the 0.5 s stimulus plus 0.5 s of post-stimulus — from
   `src.definitions.constants.AssrEpoch`, capped by the shortest inter-onset gap so
   no epoch touches a neighbouring stimulus (edge windows that don't fit the
   recording are skipped).
2. **Channel PCA.** Treat each `(freq, time)` bin as an observation and each
   channel as a variable. Fit PCA over the `(n_freqs*n_times, n_channels)` matrix
   and keep each retained component's **score map**, reshaped to
   `(n_freqs, n_times)`, plus its channel **loading** (the component's scalp
   topography). This reduces the channel dimension to a few spatial modes.
3. **Polarity alignment across participants**, done **per component and
   independently** (each component's sign is its own arbitrary choice). Each
   subject's loading is aligned to a common template and the overall orientation
   anchored to a reference channel (`src.analysis.pca_polarity`) — the same
   convention as `assr_raw_pca_analysis.ipynb`, and independent of whether a 40 Hz
   response is present.
4. **Plot** each component's TF map, per participant and averaged over
   participants, then its **scalp topography** — the loadings say *where* each
   component comes from, which is what tells you whether the channel reduction kept
   signal or isolated an artefact, and what it discarded.
5. **Compare** the components on explained variance, topography consistency and
   ASSR-band response.

The stimulus alignment (see `00-preprocessing/stimulus_alignment.ipynb`) splices
every recording so each onset lands at the **same sample index** in all
participants — one shared onsets array serves every subject.

> **Component identity is per subject, not global.** Each participant gets their
> own PCA, so "PC2" means "that subject's second-most-variance mode" — subjects can
> disagree about what that mode is. The topography-consistency diagnostic is the
> check; a later component whose consistency collapses is not one shared mode, and
> its group map is not interpretable however clean the individual maps look.

> **Small-subset notebook by design.** The wavelet file is ~49 GB, read **lazily**
> — only the selected subjects are decompressed, and each channel block is reduced
> to its small `(n_freqs, win)` trial average on the fly (all channels are kept
> because PCA needs them). Keep `SUBJECT_INDICES` to the *lowest* indices: the
> streaming reader scans from the start, so `[0, 1, 2]` is far cheaper than
> `[0, 7, 14]`. Extracting more components costs nothing extra here — the read
> dominates, and the PCA runs once per subject regardless.

> With only a handful of subjects both the group average and the template that
> aligns the signs are fragile: check the **consistency diagnostic** printed by the
> polarity step before reading any group map. This bites harder for later
> components, whose spatial modes are less consistent to begin with.

## Configuration

In [ ]:
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO   # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR

# ── Subjects to reduce (all channels are used by the PCA) ─────────────────────
# Subject indices into the concatenated array (CONCATENATED_PERSON_INDEX). Keep
# these LOW — the lazy reader scans from the start, so [0, 1, 2] only decompresses
# the first 3 of 15 subject blocks.
SUBJECT_INDICES = [3, 4, 5,]

# ── Components to extract ─────────────────────────────────────────────────
# Number of leading channel-PCA components kept per participant. All of them are
# processed IDENTICALLY (polarity alignment, diagnostics, plots), so a later
# component can be compared against PC1 instead of being assumed uninformative —
# the highest-variance mode is not necessarily the best 40 Hz carrier. Costs
# nothing extra to read: the PCA runs once per subject either way. Set to 1 to
# recover the original PC1-only behaviour.
N_COMPONENTS = 3

# ── Stimulus-locked epoch window ─────────────────────────────────────
# Taken from the paradigm definition (src.definitions.constants.AssrEpoch) so
# every onset-locked ASSR analysis cuts the same window:
#   PRE_PAD_S  = 0.1 s baseline before onset
#   POST_PAD_S = 1.0 s after onset = 0.5 s stimulus + 0.5 s post-stimulus
# The post length is capped by the shortest inter-onset gap in the subset cell, so
# no epoch can reach a neighbouring stimulus.
PRE_PAD_S = AssrEpoch.PRE_ONSET_S
POST_PAD_S = AssrEpoch.POST_ONSET_S

ASSR_FREQ = 40.0           # expected steady-state frequency (Hz)
ASSR_HALFWIDTH_HZ = 2.0    # half-width of the ASSR band used by the comparison
SFREQ = 250.0              # sampling rate of the concatenated / wavelet data

# Z-score each channel's per-frequency power against the WHOLE recording before
# epoching (matches the inspection notebook): removes the 1/f tilt so the PCA is
# not dominated by absolute low-frequency power. Set False to run PCA on raw power.
ZSCORE_VS_RECORDING = True

# ── Wavelet cache descriptor (matches the stored filename) ─────────────────
WAVELET_FREQ_SIG = "1.000_50.000_50"   # freqs[0]_freqs[-1]_n_freqs
N_WAVELET_FREQS = 50

# ── Plot saving ────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_wavelet_pca"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Group          : {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Subjects       : {SUBJECT_INDICES}")
print(f"Components     : {N_COMPONENTS} (PC1..PC{N_COMPONENTS})")
print(f"Pre-onset pad  : {PRE_PAD_S} s")
print(f"Post-onset span: {POST_PAD_S} s "
      f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
      f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)")
print(f"Z-score vs rec : {ZSCORE_VS_RECORDING}")
print(f"Plots -> {PLOTS_DIR}")

## Load Concatenated Onsets, Metadata & Channel Names

In [ ]:
safe_label = f"{CONDITION.value}_{MUSIC_TYPE.value}"

concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
onsets_path = concat_dir / f"{safe_label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
meta_path = concat_dir / f"{safe_label}.metadata.csv"

wavelet_path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / "wavelets"
    / "broadband"
    / f"{safe_label}__wavelet_power__{WAVELET_FREQ_SIG}__freqdim1.npz"
)
for p in (onsets_path, meta_path, wavelet_path):
    assert p.exists(), f"Missing expected file: {p}"

onsets = np.load(onsets_path)                        # (n_onsets,) shared sample idx
meta = pd.read_csv(meta_path, index_col=0)
gaps = np.diff(onsets)
print(f"Stimulus onsets : {onsets.shape}  range [{onsets.min()}, {onsets.max()}]")
print(f"Inter-onset gap : min {int(gaps.min())}, median {int(np.median(gaps))}, "
      f"max {int(gaps.max())} samples")

# Channel names are stored in the wavelet feature_names (layout: channel*N_FREQS).
with zipfile.ZipFile(wavelet_path) as _z:
    with _z.open("feature_names.npy") as _f:
        feature_names = npformat.read_array(_f, allow_pickle=True)
channel_names = [str(feature_names[i * N_WAVELET_FREQS]).split("@")[0]
                 for i in range(len(feature_names) // N_WAVELET_FREQS)]
n_channels = len(channel_names)
print(f"Channels parsed : {n_channels} (first={channel_names[0]}, "
      f"last={channel_names[-1]})")
meta[["SingleDataMetadata.PARTICIPANT_ID", "SingleDataMetadata.CONDITION",
      "SingleDataMetadata.CONCATENATED_PERSON_INDEX"]].head()

## Subset Selection & Epoch Window

The epoch comes from the **paradigm**, not from the observed jitter: `PRE_PAD_S` = 0.1 s
of baseline before onset, then `POST_PAD_S` = 1.0 s after onset — the **0.5 s stimulus
plus 0.5 s post-stimulus**, so a response outlasting the stimulus stays visible. Both
values live in `src.definitions.constants.AssrEpoch`, shared with
`assr_raw_pca_analysis.ipynb` and the onset-locked IVA quality references.

The post-onset length is then **capped** by the shortest inter-onset gap, so no epoch can
reach a neighbouring stimulus. With ~1.25 s between onsets the full 1.1 s epoch fits
inside one inter-onset interval — including the pre-onset baseline, which the earlier
`gaps.min()`-derived window did not.

`stim_mask` marks the driven interval. It matters here: a 1 s epoch around a 0.5 s
stimulus is half silence, so any steady-state estimate averaged over the whole post-onset
window is diluted by the silent half.

In [ ]:
# Epoch window in samples: paradigm span, capped by the shortest inter-onset gap.
PRE = int(round(PRE_PAD_S * SFREQ))
POST = min(int(round(POST_PAD_S * SFREQ)), int(gaps.min()))
epoch_times = np.arange(-PRE, POST) / SFREQ         # (win,) seconds, t=0 at onset
stim_mask = AssrEpoch.stimulus_mask(epoch_times)    # driven interval, for later use

# Map subject indices -> participant labels via the concatenated metadata.
pidx_col = "SingleDataMetadata.CONCATENATED_PERSON_INDEX"
pid_col = "SingleDataMetadata.PARTICIPANT_ID"
idx_to_pid = dict(zip(meta[pidx_col], meta[pid_col].astype(str).str.zfill(3)))
# Order subjects by participant ID so every per-participant plot (and the loaded
# trial_avg stack, built from this order) is sorted by PID rather than by
# concatenated index.
SUBJECT_INDICES = sorted(
    SUBJECT_INDICES, key=lambda si: int(idx_to_pid.get(si, "9999"))
)
subject_labels = [f"PSI{idx_to_pid.get(si, '???')}" for si in SUBJECT_INDICES]

print(f"Selected subjects: {dict(zip(SUBJECT_INDICES, subject_labels))}")
print(f"Epoch window     : {PRE + POST} samples ({PRE} pre, {POST} post) "
      f"= [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
print(f"Stimulus interval: [0.000, {AssrEpoch.STIMULUS_DURATION_S:.3f}] s "
      f"({int(stim_mask.sum())} samples); post-stimulus "
      f"{epoch_times[-1] - AssrEpoch.STIMULUS_DURATION_S:.3f} s")
if POST < int(round(POST_PAD_S * SFREQ)):
    print(f"  NOTE: post-onset span trimmed from {POST_PAD_S} s to "
          f"{POST / SFREQ:.3f} s by the shortest inter-onset gap "
          f"({int(gaps.min())} samples).")
if PRE + POST > int(gaps.min()):
    print(f"  WARNING: epoch ({PRE + POST} samples) exceeds the shortest gap "
          f"({int(gaps.min())}) — the baseline reaches into the previous stimulus.")

## Helpers — epoching & lazy trial-averaging reader

In [ ]:
def epoch_average(arr, onsets, pre, post):
    """Average fixed windows around each onset along the LAST axis.

    Args:
        arr: array whose last axis is time, e.g. ``(n_freqs, n_times)``.
        onsets: stimulus onset sample indices.
        pre, post: samples kept before / after each onset (window = pre + post).

    Returns:
        ``(averaged, n_used)`` where the time axis is replaced by the
        ``pre + post`` window, averaged over every onset whose window fits inside
        the recording (edge windows are skipped).
    """
    n_time = arr.shape[-1]
    acc = None
    n_used = 0
    for o in onsets:
        s, e = o - pre, o + post
        if s < 0 or e > n_time:
            continue
        seg = arr[..., s:e]
        acc = seg.astype(np.float64) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the recording.")
    return acc / n_used, n_used


def load_wavelet_trial_averaged(npz_path, subject_indices, n_channels, n_freqs,
                                onsets, pre, post, *, zscore_time=True):
    """Lazily read a huge wavelet npz and trial-average EVERY channel per subject.

    ``savez_compressed`` stores ``data.npy`` as a single deflate stream, so it
    must be decompressed sequentially. This reader streams one
    ``(n_freqs, n_times)`` channel block at a time and immediately reduces it to
    its small ``(n_freqs, win)`` stimulus-locked trial average, so the 49 GB file
    never lands in RAM (peak = one block, ~19 MB). All channels are kept because
    the channel-wise PCA needs the full electrode set.

    Args:
        zscore_time: z-score each channel's per-frequency power against the whole
            recording before epoching (removes the 1/f tilt).

    Returns:
        ``(data, n_times, n_used)`` where ``data`` has shape
        ``(len(subject_indices), n_channels, n_freqs, pre + post)`` and ``n_used``
        maps subject idx -> number of stimuli averaged.
    """
    subj_set = set(subject_indices)
    max_subj = max(subject_indices)
    win = pre + post
    collected = {si: np.empty((n_channels, n_freqs, win)) for si in subject_indices}
    n_used = {}
    with zipfile.ZipFile(npz_path) as z:
        with z.open("data.npy") as f:
            ver = npformat.read_magic(f)
            if ver == (1, 0):
                shape, _, dtype = npformat.read_array_header_1_0(f)
            else:
                shape, _, dtype = npformat.read_array_header_2_0(f)
            n_subj_total, n_feat_flat, n_times = shape
            assert n_feat_flat == n_channels * n_freqs, (
                f"feature axis {n_feat_flat} != n_channels*n_freqs "
                f"{n_channels * n_freqs}"
            )
            block_bytes = n_freqs * n_times * dtype.itemsize  # one channel block
            for s in range(max_subj + 1):
                for c in range(n_channels):
                    buf = f.read(block_bytes)
                    if s not in subj_set:
                        continue
                    block = np.frombuffer(
                        buf, dtype=dtype, count=n_freqs * n_times
                    ).reshape(n_freqs, n_times)
                    if zscore_time:
                        block = zscore(block, axis=1)
                    ev, used = epoch_average(block, onsets, pre, post)  # (f, win)
                    collected[s][c] = ev
                    n_used[s] = used
                if s in subj_set:
                    print(f"  scanned subject block {s} ...")
    data = np.stack([collected[si] for si in subject_indices])
    return data, n_times, n_used


print("Helpers defined.")

## Load & Trial-Average the Wavelet Power

Stream the selected subjects out of the 49 GB cache, reducing every channel to its
stimulus-locked trial average on the fly. This decompresses sequentially from the
start of the file (progress printed per subject block); the result is a small
`(n_subj, n_channels, n_freqs, win)` array.

In [ ]:
wavelet_freqs = np.arange(1.0, N_WAVELET_FREQS + 1.0)   # 1..50 Hz (as cached)

trial_avg, n_times_wav, n_used_w = load_wavelet_trial_averaged(
    wavelet_path,
    subject_indices=SUBJECT_INDICES,
    n_channels=n_channels,
    n_freqs=N_WAVELET_FREQS,
    onsets=onsets,
    pre=PRE,
    post=POST,
    zscore_time=ZSCORE_VS_RECORDING,
)
print(f"Trial-averaged wavelet: {trial_avg.shape}  (n_times_full={n_times_wav})")
for si, label in zip(SUBJECT_INDICES, subject_labels):
    print(f"  {label}: averaged {n_used_w[si]} stimuli")

## Channel PCA — leading component TF maps (per subject)

For each subject the trial-averaged data is `(n_channels, n_freqs, win)`. We reshape
it so each `(freq, time)` bin is an **observation** and each channel is a
**variable** — matrix `X` of shape `(n_freqs*win, n_channels)` — and fit PCA. Each
of the first `N_COMPONENTS` components' **scores**, reshaped to `(n_freqs, win)`,
is the TF map of one spatial mode; its **loading** `(n_channels,)` is that
component's scalp topography, and it is what fixes the sign.

Every component goes through the **exact same** treatment; nothing downstream is
special-cased for PC1. PCA orders components by explained *channel variance* across
the whole TF plane, which is not the quantity of interest: the plane is dominated
by the broad low-frequency bins, so a slow onset response can own PC1 while the
narrow 40 Hz steady-state lands in PC2 or PC3. The comparison cell scores every
component on the same measures before any of them is dismissed.

**Polarity alignment across participants, per component.** PCA component signs are
arbitrary and each component is flipped independently of the others, so alignment
runs **separately per component**. Each subject's loading is aligned to a common
template (iteratively-refined group-mean loading) and the overall orientation
anchored to a reference channel (`src.analysis.pca_polarity`) — the same convention
as `assr_raw_pca_analysis.ipynb`. This maximises cross-subject agreement, which is
what makes the group averages below meaningful; the TF map is flipped with its
loading so map and topography stay consistent.

This replaces an earlier convention that referenced the **ASSR band**
(`ASSR_FREQ ± 2` Hz) in the post-onset interval. That rule is only as reliable as the
40 Hz response itself: when the response is absent the reference value is noise, each
subject's sign becomes a coin flip, and the group average collapses toward
`1/sqrt(n_subjects)` of the aligned amplitude. Aligning on the loading is well
defined either way — and it has to be, because for later components a 40 Hz
reference would be noise by construction.

The explained-variance ratio is reported per component, together with a
**consistency diagnostic** for that component's alignment.

In [ ]:
n_freqs = N_WAVELET_FREQS
win = PRE + POST
subs = list(SUBJECT_INDICES)
n_subj = len(subs)
pc_labels = [f"PC{c + 1}" for c in range(N_COMPONENTS)]

# Component-major storage: axis 0 = component, axis 1 = subject (in the order of
# `subs` / `subject_labels`). Every component is filled by the SAME code path.
tf_maps = np.empty((N_COMPONENTS, n_subj, n_freqs, win))   # score maps
loadings = np.empty((N_COMPONENTS, n_subj, n_channels))    # channel topographies
explained = np.empty((N_COMPONENTS, n_subj))               # explained var. ratio

for k, si in enumerate(subs):
    ep = trial_avg[k]                              # (n_channels, n_freqs, win)
    # Observations = (freq, time) bins, variables = channels.
    X = ep.reshape(n_channels, n_freqs * win).T    # (n_freqs*win, n_channels)
    pca = PCA(n_components=N_COMPONENTS)
    scores = pca.fit_transform(X)                  # (n_freqs*win, n_pc)
    tf_maps[:, k] = scores.T.reshape(N_COMPONENTS, n_freqs, win)
    loadings[:, k] = pca.components_               # (n_pc, n_channels)
    explained[:, k] = pca.explained_variance_ratio_

# ── Align polarity ACROSS participants, INDEPENDENTLY PER COMPONENT ───────
# Flip every subject's loading toward a common template (iteratively-refined
# group-mean loading), which MAXIMISES cross-subject topography agreement, then
# anchor the overall orientation to a reference channel. Each component is flipped
# independently of the others, so each gets its own pass. The TF map is flipped
# with its loading. This replaces an ASSR-band sign reference: that rule is only as
# reliable as the 40 Hz response itself, and when the response is absent its
# reference value is noise, making each subject's sign a coin flip.
consistency = []   # per component: SignConsistency of the aligned loadings
n_flipped = []     # per component: how many subjects were sign-flipped
anchors = []       # per component: channel that fixed the overall orientation
for c in range(N_COMPONENTS):
    signs, anchor = align_pc1_signs(loadings[c], channel_names=channel_names)
    loadings[c] = apply_pc1_signs(loadings[c], signs)
    tf_maps[c] = apply_pc1_signs(tf_maps[c], signs)
    # Diagnostic: after alignment every subject should correlate POSITIVELY with
    # the group-mean topography. One that does not is a topographic outlier, not a
    # sign problem — averaging it into the group map cancels real signal.
    consistency.append(topography_consistency(loadings[c]))
    n_flipped.append(int((signs < 0).sum()))
    anchors.append(anchor)

print(f"Channel PCA over {n_channels} channels, {N_COMPONENTS} component(s) kept, "
      f"{n_subj} participants.")
for c, pc in enumerate(pc_labels):
    cons = consistency[c]
    print(f"\n{pc}  (mean explained variance {explained[c].mean() * 100:.1f}%, "
          f"range {explained[c].min() * 100:.1f}–{explained[c].max() * 100:.1f}%)")
    print(f"  polarity aligned      : {n_flipped[c]} subject(s) flipped; "
          f"anchor {anchors[c]}")
    print(f"  topographies agreeing : {cons.n_agreeing}/{cons.n_subjects}")
    print(f"  median pairwise r     : {cons.median_pairwise_r:+.3f}")
    print(f"  weakest subject r     : {cons.min_subject_r:+.3f}")
    if cons.n_agreeing < cons.n_subjects:
        print(f"  WARNING: {cons.n_subjects - cons.n_agreeing} subject(s) still "
              f"anti-correlate with the group topography — inspect before "
              f"trusting the {pc} group map.")
    print("  per-subject explained variance: " + ", ".join(
        f"{label} {explained[c, k] * 100:.1f}%"
        for k, label in enumerate(subject_labels)
    ))

### Per-Participant TF Maps — one figure per component

One figure per component, one panel per participant, all drawn by the same code and
saved separately (`wavelet_pca_pc{n}_tf_per_participant.png`). Each figure keeps its
own symmetric colour scale: component scores shrink with component order by
construction, so a scale shared across components would flatten PC2/PC3 into uniform
grey and hide the very structure being compared. Panels **within** a figure share a
scale and stay comparable across participants.

In [ ]:
extent = [epoch_times[0], epoch_times[-1], wavelet_freqs[0], wavelet_freqs[-1]]

for c, pc in enumerate(pc_labels):
    # Symmetric colour scale shared by the panels of THIS component only.
    vmax = float(np.abs(tf_maps[c]).max())
    if vmax == 0.0:
        vmax = 1e-12

    fig, axes = plt.subplots(1, n_subj, figsize=(5 * n_subj, 4.2), squeeze=False)
    for ax, k, label in zip(axes[0], range(n_subj), subject_labels):
        im = ax.imshow(tf_maps[c, k], aspect="auto", origin="lower", extent=extent,
                       cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.axvline(0.0, color="k", ls="--", lw=0.8)
        ax.axhline(ASSR_FREQ, color="green", ls=":", lw=1.0)
        ax.set_title(f"{label}  ({pc} {explained[c, k] * 100:.0f}%)")
        ax.set_xlabel("Time rel. onset (s)")
        ax.set_ylabel("Frequency (Hz)")
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.85, label=f"{pc} score (a.u.)")
    fig.suptitle(
        f"Per-participant channel-PCA {pc} TF map — "
        f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj}, "
        f"mean EV {explained[c].mean() * 100:.0f}%, agreeing "
        f"{consistency[c].n_agreeing}/{consistency[c].n_subjects})",
        y=1.02,
    )
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / f"wavelet_pca_{pc.lower()}_tf_per_participant.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

### Component TF Maps — averaged over participants

Mean of the per-subject maps, one panel per component. Polarity is aligned across
participants within each component (loadings matched to a common template), so each
average is meaningful wherever that component's spatial mode is consistent across
subjects — check the per-component **consistency diagnostic** printed by the
polarity step before reading a panel. Each panel is scaled to its own maximum, so
compare *structure* across panels, not intensity; the numbers in the comparison
cell are what compare intensity.

In [ ]:
group_map = tf_maps.mean(axis=1)               # (n_pc, n_freqs, win)

fig, axes = plt.subplots(
    1, N_COMPONENTS, figsize=(7 * N_COMPONENTS, 4.6), squeeze=False
)
for c, pc in enumerate(pc_labels):
    ax = axes[0, c]
    gvmax = float(np.abs(group_map[c]).max())
    if gvmax == 0.0:
        gvmax = 1e-12
    im = ax.imshow(group_map[c], aspect="auto", origin="lower", extent=extent,
                   cmap="RdBu_r", vmin=-gvmax, vmax=gvmax)
    ax.axvline(0.0, color="k", ls="--", lw=0.8, label="onset")
    ax.axhline(ASSR_FREQ, color="green", ls=":", lw=1.2, label=f"{ASSR_FREQ:.0f} Hz")
    ax.set_title(
        f"{pc} — mean EV {explained[c].mean() * 100:.0f}%, agreeing "
        f"{consistency[c].n_agreeing}/{consistency[c].n_subjects}, median r "
        f"{consistency[c].median_pairwise_r:+.2f}",
        fontsize=10,
    )
    ax.set_xlabel("Time relative to onset (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.legend(loc="upper right", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.9, label=f"{pc} score (a.u.)")
fig.suptitle(
    f"Channel-PCA component TF maps, averaged over participants — "
    f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj})",
    y=1.03,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "wavelet_pca_tf_group_average.png", dpi=150,
                bbox_inches="tight")
plt.show()

## Topomap Montage — electrode positions

Electrode positions come from one `RAW_CROPPED` recording's `Info`, remapped onto
the canonical `channel_names` order that the wavelet cache uses — the same
construction as `assr_raw_pca_analysis.ipynb`. Loading one recording header with
`preload=False` reads no sample data.

In [ ]:
topo_handler = DatasetHandler(EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
info_fname = meta.loc[
    meta[pidx_col] == SUBJECT_INDICES[0], "SingleDataMetadata.FILENAME"
].iloc[0]
topo_info = (
    topo_handler.load_data_file(
        info_fname,
        is_processed=True,
        processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
        preload=False,
    )
    .pick("eeg")
    .info
)
name_pos = {n: i for i, n in enumerate(channel_names)}
assert set(topo_info["ch_names"]) <= set(name_pos), (
    "RAW_CROPPED channels are not a subset of the wavelet channel names."
)
info_order = [name_pos[n] for n in topo_info["ch_names"]]
print(f"Topomap montage: {len(topo_info['ch_names'])} electrodes "
      f"(reordered onto the wavelet channel order).")

### Component Topographies — what the channel reduction kept, and what it dropped

The TF maps above show *when and at which frequency* each component is active; they
say nothing about *where on the scalp* it comes from. That is the loading vector, and
it is the thing to check before trusting the reduction: collapsing 195 channels onto
`N_COMPONENTS` maps throws away every spatial pattern orthogonal to these
topographies, so these panels are the record of what survived.

One figure per component, per participant plus the group average, following the
notebook-05 convention — signed values, diverging `RdBu_r`, symmetric colour scale.
Loadings are unit-norm, so magnitudes are comparable across components; each figure
still takes its own scale from its own 99th percentile.

Reading them for what was filtered out:

- **A plausible evoked topography** (fronto-central dipole around Cz for the ASSR)
  means the component is carrying signal, and its TF map is worth reading.
- **A component concentrated on one or two electrodes, or on the rim,** is a bad
  channel or an edge artefact that the PCA has isolated — which is useful in itself:
  it is then *not* contaminating the other components' maps.
- **A smooth gradient across the whole scalp** is usually a reference or drift mode.
- **A topography that differs wildly between participants** (flagged by the
  consistency diagnostic) means "PC*k*" is not the same thing in each subject, so the
  group panel for it averages unlike maps.

The cell also prints how much channel variance the kept components retain and how
much the reduction discarded — the blunt version of "what has been filtered out".
A large discarded fraction is not automatically a problem (much of it is noise
spread across electrodes), but it bounds what any later analysis on these maps can
possibly see.

In [ ]:
group_loading = loadings.mean(axis=1)          # (n_pc, n_channels)

# How much channel variance survives the reduction, per participant.
retained = explained.sum(axis=0)               # (n_subj,) summed over kept PCs
print(f"Channel variance retained by the {N_COMPONENTS} kept component(s): "
      f"mean {retained.mean() * 100:.1f}% "
      f"(range {retained.min() * 100:.1f}–{retained.max() * 100:.1f}%); "
      f"discarded mean {(1 - retained.mean()) * 100:.1f}%.")
for k, label in enumerate(subject_labels):
    print(f"  {label}: kept {retained[k] * 100:.1f}%, "
          f"discarded {(1 - retained[k]) * 100:.1f}%")

for c, pc in enumerate(pc_labels):
    panels = [
        (f"{subject_labels[k]}  ({pc} {explained[c, k] * 100:.0f}%)",
         loadings[c, k][info_order])
        for k in range(n_subj)
    ]
    panels.append((f"group average  (mean {pc} {explained[c].mean() * 100:.0f}%)",
                   group_loading[c][info_order]))

    # Symmetric shared scale (nb05 convention): 99th percentile of |loading|.
    vlim = float(np.percentile(np.abs(np.concatenate([v for _, v in panels])), 99))
    if vlim == 0.0:
        vlim = 1e-12

    n_panels = len(panels)
    ncols = min(5, n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.9 * nrows),
                             squeeze=False)
    flat = axes.flatten()
    im = None
    for ax, (label, vals) in zip(flat, panels):
        im, _ = mne.viz.plot_topomap(
            vals, topo_info, axes=ax, show=False, cmap="RdBu_r",
            vlim=(-vlim, vlim), contours=4,
        )
        ax.set_title(label, fontsize=9)
    for ax in flat[n_panels:]:
        ax.axis("off")
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6,
                 label=f"{pc} loading (a.u.)")
    fig.suptitle(
        f"Per-participant channel-PCA {pc} topography — "
        f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj}, agreeing "
        f"{consistency[c].n_agreeing}/{consistency[c].n_subjects}, median r "
        f"{consistency[c].median_pairwise_r:+.2f})",
        y=1.0,
    )
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / f"wavelet_pca_{pc.lower()}_topomap_per_participant.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

## Component Comparison — is PC1 actually the best one?

Every component was extracted, aligned and plotted identically, so they can be
scored side by side. PCA ranks them by explained **channel variance** over the whole
TF plane, which is not the quantity of interest: the plane is mostly low-frequency
bins, so a broad onset response can own PC1 while the narrow 40 Hz steady-state
sits in PC2 or PC3. The table reports, per component:

- **mean EV** — mean explained variance ratio across participants (what PCA
  actually maximised).
- **agreeing / median r / min r** — the topography-consistency diagnostic on the
  aligned loadings. This is the gate: a component whose subjects disagree has no
  interpretable group map, however clean the individual maps look. With a handful
  of subjects these numbers are noisy — read them as a veto, not a ranking.
- **ASSR contrast** — mean score in the `ASSR_FREQ ± ASSR_HALFWIDTH_HZ` band during
  the **driven interval** minus the same band's pre-onset baseline, divided by the
  map's own standard deviation. The normalisation is what makes components with
  different absolute score magnitudes comparable; the baseline subtraction is what
  keeps a component that simply sits high at 40 Hz all epoch from scoring.
- **broadband contrast** — the same driven-minus-baseline contrast over **all**
  frequencies. Read it next to the ASSR column: a component whose two contrasts are
  similar responds to the stimulus broadly (an onset/arousal response), whereas one
  where the ASSR column clearly exceeds the broadband column is frequency-specific
  — the actual steady-state signature.

Both contrasts are given as the per-subject median and as the value of the
group-mean map. Group above per-subject means the effect is consistent enough to
survive averaging; group well below means subjects each have something at 40 Hz but
not in the same place on the TF plane.

The pre-onset baseline is only `PRE` samples (0.1 s), so these contrasts are
coarse — enough to rank components, not to quantify a response.

A later component wins only if it beats PC1 on the ASSR contrast **and** holds up
on topography consistency. Beating it on contrast while failing consistency means
subjects have 40 Hz in some second mode, but not in the *same* second mode.

In [ ]:
band_mask = np.abs(wavelet_freqs - ASSR_FREQ) <= ASSR_HALFWIDTH_HZ
base_mask = epoch_times < 0.0          # pre-onset baseline samples
assert band_mask.any() and base_mask.any() and stim_mask.any()


def driven_contrast(tf_map, freq_mask):
    """Driven-minus-baseline score contrast, in units of the map's own SD.

    Args:
        tf_map: ``(n_freqs, win)`` component score map.
        freq_mask: boolean mask over frequencies to average within.

    Returns:
        Mean score inside ``freq_mask`` during the stimulus, minus the same band's
        pre-onset baseline, divided by the map's standard deviation — dimensionless,
        so components with different score magnitudes stay comparable.
    """
    sd = float(tf_map.std())
    if sd == 0.0:
        return float("nan")
    band = tf_map[freq_mask]
    return float(band[:, stim_mask].mean() - band[:, base_mask].mean()) / sd


all_freqs = np.ones_like(band_mask, dtype=bool)
subject_assr = np.array([[driven_contrast(tf_maps[c, k], band_mask)
                          for k in range(n_subj)] for c in range(N_COMPONENTS)])
subject_broad = np.array([[driven_contrast(tf_maps[c, k], all_freqs)
                           for k in range(n_subj)] for c in range(N_COMPONENTS)])
group_assr = np.array([driven_contrast(group_map[c], band_mask)
                       for c in range(N_COMPONENTS)])
group_broad = np.array([driven_contrast(group_map[c], all_freqs)
                        for c in range(N_COMPONENTS)])

comparison = pd.DataFrame({
    "component": pc_labels,
    "mean_EV_%": [explained[c].mean() * 100 for c in range(N_COMPONENTS)],
    "agreeing": [f"{consistency[c].n_agreeing}/{consistency[c].n_subjects}"
                 for c in range(N_COMPONENTS)],
    "median_pairwise_r": [consistency[c].median_pairwise_r
                          for c in range(N_COMPONENTS)],
    "min_subject_r": [consistency[c].min_subject_r for c in range(N_COMPONENTS)],
    "assr_contrast_subject_median": np.nanmedian(subject_assr, axis=1),
    "assr_contrast_group_map": group_assr,
    "broadband_contrast_subject_median": np.nanmedian(subject_broad, axis=1),
    "broadband_contrast_group_map": group_broad,
}).set_index("component")

selectivity = group_assr - group_broad
best_assr = pc_labels[int(np.nanargmax(group_assr))]
best_sel = pc_labels[int(np.nanargmax(selectivity))]
best_cons = pc_labels[int(np.argmax([c.median_pairwise_r for c in consistency]))]
print(f"ASSR band: {wavelet_freqs[band_mask].min():.0f}–"
      f"{wavelet_freqs[band_mask].max():.0f} Hz ({int(band_mask.sum())} bins); "
      f"driven {int(stim_mask.sum())} samples vs baseline "
      f"{int(base_mask.sum())} samples.")
print(f"Highest ASSR-band contrast on the group map : {best_assr}")
print(f"Most frequency-selective (ASSR - broadband) : {best_sel}")
print(f"Most consistent topography across subjects  : {best_cons}")
comparison.round(3)